In [1]:
using Pkg
Pkg.activate("../test")

  Activating project at `~/repos/WaveGreen2D/test`


In [2]:
using JLD2
using Test
using StaticArrays

using WaveGreen2D.Chebyshev
using WaveGreen2D.Chebyshev: normalize, contains, clenshaw, gradient_clenshaw, hessian_clenshaw

In [3]:
@testset "0-D Chebyshev series" begin
    coefs = Array{Float64, 0}(undef)
    cs = ChebyshevSeries(coefs, SVector{0}(), SVector{0}())
    
    @test cs == coefs[]
end

Test Summary:        | Pass  Total  Time
0-D Chebyshev series |    1      1  0.2s


Test.DefaultTestSet("0-D Chebyshev series", Any[], 1, false, false, true, 1.778961060825255e9, 1.778961061014604e9, false, "In[3]", Random.Xoshiro(0x7b921d91f7679ffc, 0xe7bb99c6fc402fbf, 0xdebf85c9e7fe77cc, 0xcc805a0c2d446619, 0x2404fe2db0ff50e4))

In [4]:
@testset "1-D polynomial function" begin
    coefs = collect(UnitRange(1.0, 6.0))
    cs = ChebyshevSeries(coefs, SA[-1.0], SA[1.0])
    @test clenshaw(cs, 0.5) == -3
    @test gradient_clenshaw(cs, 0.5) == (-3, -42)
    @test hessian_clenshaw(cs, 0.5) == (-3, -42, -20)
end

Test Summary:           | Pass  Total  Time
1-D polynomial function |    3      3  0.7s


Test.DefaultTestSet("1-D polynomial function", Any[], 3, false, false, true, 1.778961062325666e9, 1.778961063034414e9, false, "In[4]", Random.Xoshiro(0x7b921d91f7679ffc, 0xe7bb99c6fc402fbf, 0xdebf85c9e7fe77cc, 0xcc805a0c2d446619, 0x2404fe2db0ff50e4))

In [5]:
@testset "2-D polynomial function" begin
    coefs = Float64[3*i*(j+1) for i in 1:6, j in 1:5]
    cs = ChebyshevSeries(coefs, SA[-1.0, -1.0], SA[1.0, 1.0])
    @test clenshaw(cs, SA[-0.5, 0.5]) == 58.5
    @test gradient_clenshaw(cs, SA[-0.5, 0.5]) == (58.5, 273, 117)
    @test hessian_clenshaw(cs, SA[-0.5, 0.5]) == (58.5, 273, -2418, 117, 546, -1116)
end

Test Summary:           | Pass  Total  Time
2-D polynomial function |    3      3  2.0s


Test.DefaultTestSet("2-D polynomial function", Any[], 3, false, false, true, 1.778961063123067e9, 1.778961065137052e9, false, "In[5]", Random.Xoshiro(0x7b921d91f7679ffc, 0xe7bb99c6fc402fbf, 0xdebf85c9e7fe77cc, 0xcc805a0c2d446619, 0x2404fe2db0ff50e4))

In [6]:
@testset "3-D polynomial function" begin
    coefs = Float64[0.25*(i-1)*(j+1)*(k+3) for i in 0:6, j in 0:4, k in 0:5]
    cs = ChebyshevSeries(coefs, SA[-1.0, -1.0, -1.0], SA[1.0, 1.0, 1.0])
    @test clenshaw(cs, SA[-0.5, 0.5, -0.5]) == 9
    @test gradient_clenshaw(cs, SA[-0.5, 0.5, -0.5]) == (9, -45, 18, 54)
    @test hessian_clenshaw(cs, SA[-0.5, 0.5, -0.5]) == (9, -45, -702, 18, -90, -150, 54, -270, 108, -492)
end

Test Summary:           | Pass  Total  Time
3-D polynomial function |    3      3  3.4s


Test.DefaultTestSet("3-D polynomial function", Any[], 3, false, false, true, 1.77896106527688e9, 1.778961068659005e9, false, "In[6]", Random.Xoshiro(0x7b921d91f7679ffc, 0xe7bb99c6fc402fbf, 0xdebf85c9e7fe77cc, 0xcc805a0c2d446619, 0x2404fe2db0ff50e4))

In [7]:
@testset "1-D transcedental function" begin
    # Coefficients for f(x) = sin(x)
    coefs = [6.0219470125554653e-01, 5.1362516667910696e-01, -1.0354634426296377e-01, -1.3732034234358511e-02, 1.3586698380903617e-03, 1.0726309440600129e-04, -7.0462967934650502e-06, -3.9639025062457094e-07, 1.9499597787191747e-08, 8.5229287361514171e-10, -3.3516569268963781e-11, -1.1979172283756631e-12, 3.9247734881581288e-14, 1.1922754450388594e-15]
    cs = ChebyshevSeries(coefs, SA[0.0], SA[π/2])
    
    x = π/3
    
    @test cs(x) ≈ sin(x)
    
    y, dy_dx = gradient(cs, x)
    @test y ≈ sin(x)
    @test dy_dx ≈ cos(x)

    y, dy_dx, d²y_dx² = hessian(cs, x)
    @test y ≈ sin(x)
    @test dy_dx ≈ cos(x)
    @test d²y_dx² ≈ -sin(x)
end

Test Summary:              | Pass  Total  Time
1-D transcedental function |    6      6  0.5s


Test.DefaultTestSet("1-D transcedental function", Any[], 6, false, false, true, 1.778961068668973e9, 1.778961069181258e9, false, "In[7]", Random.Xoshiro(0x7b921d91f7679ffc, 0xe7bb99c6fc402fbf, 0xdebf85c9e7fe77cc, 0xcc805a0c2d446619, 0x2404fe2db0ff50e4))

In [8]:
@testset "1-D transcedental function" begin
    # Coefficients for f(x) = sin(x) in [0.0, π/2]
    coefs = [
        6.0219470125554653e-01, 5.1362516667910696e-01, -1.0354634426296377e-01,
        -1.3732034234358511e-02, 1.3586698380903617e-03, 1.0726309440600129e-04,
        -7.0462967934650502e-06, -3.9639025062457094e-07, 1.9499597787191747e-08,
        8.5229287361514171e-10, -3.3516569268963781e-11, -1.1979172283756631e-12,
        3.9247734881581288e-14, 1.1922754450388594e-15,
    ]
    cs = ChebyshevSeries(coefs, SA[0.0], SA[π/2])
    x = π / 3

    @test cs(x) ≈ sin(x)

    y, dy_dx = gradient(cs, x)
    @test y ≈ sin(x)
    @test dy_dx ≈ cos(x)

    y, dy_dx, d²y_dx² = hessian(cs, x)
    @test y ≈ sin(x)
    @test dy_dx ≈ cos(x)
    @test d²y_dx² ≈ -sin(x)
end

Test Summary:              | Pass  Total  Time
1-D transcedental function |    6      6  0.0s


Test.DefaultTestSet("1-D transcedental function", Any[], 6, false, false, true, 1.778961069191753e9, 1.77896106919185e9, false, "In[8]", Random.Xoshiro(0x7b921d91f7679ffc, 0xe7bb99c6fc402fbf, 0xdebf85c9e7fe77cc, 0xcc805a0c2d446619, 0x2404fe2db0ff50e4))

In [9]:
@testset "2-D transcedental function" begin
    # Coefficients for f(x) = cos(x*y/4) in [-0.05, 0.2]×[0.15, 0.4]
    coefs = reshape([
        9.9997773458446393e-01, -2.9687147218652036e-05, -1.4843529511822712e-05, 8.8195057654251603e-11, 2.2048639029150242e-11, -1.4062273866127676e-05, -1.8749619143127693e-05, -9.3747619646810078e-06, 9.5213854486690572e-11, 2.3803294887226696e-11, -1.1718218490754714e-06, -1.5624104826702735e-06, -7.8119405174505616e-07, 2.2379192480056440e-11, 5.5947397997171135e-12, 5.7982154469572348e-12, 9.7654339308064030e-12, 6.1033801987253946e-12, 2.4413393152714729e-12, 6.1031898895719092e-13, 2.4155138021831353e-13, 4.0687775290673887e-13, 2.5430493681875442e-13, 1.0171613218073183e-13, 2.5427286228404570e-14,
    ], 5, 5)
    
    cs = ChebyshevSeries(coefs, SA[-0.05, 0.2], SA[0.15, 0.4])
    x̄ = [0.1, 0.3]
    x, y = x̄
    
    f = cos(0.25*x*y)
    df_dx = -0.25*y*sin(0.25*x*y)
    df_dy = -0.25*x*sin(0.25*x*y)
    d²f_dx² = -0.0625*y^2*cos(0.25*x*y)
    d²f_dxy = -0.0625*(x*y*cos(0.25*x*y) + 4*sin(0.25*x*y))
    d²f_dy² = -0.0625*x^2*cos(0.25*x*y)
    
    @test cs(x̄) ≈ f

    z, ∇z = gradient(cs, x̄)
    @test z ≈ f
    @test ∇z[1] ≈ df_dx
    @test ∇z[2] ≈ df_dy

    z, ∇z, Hz = hessian(cs, x̄)
    @test z ≈ f
    @test ∇z[1] ≈ df_dx
    @test ∇z[2] ≈ df_dy
    @test Hz[1, 1] ≈ d²f_dx²
    @test Hz[1, 2] ≈ d²f_dxy
    @test Hz[2, 1] ≈ d²f_dxy
    @test Hz[2, 2] ≈ d²f_dy²
end

Test Summary:              | Pass  Total  Time
2-D transcedental function |   11     11  0.4s


Test.DefaultTestSet("2-D transcedental function", Any[], 11, false, false, true, 1.778961069208006e9, 1.778961069635086e9, false, "In[9]", Random.Xoshiro(0x7b921d91f7679ffc, 0xe7bb99c6fc402fbf, 0xdebf85c9e7fe77cc, 0xcc805a0c2d446619, 0x2404fe2db0ff50e4))

In [10]:
@testset "3-D transcedental function" begin
    # Coefficients for f(x) = exp(x*y) * cos(x + z/2) in [0.5, 0.7]×[-0.2, 0.0]×[1.0, 1.2]
    @load "../test/test_chebyshev_3dtf.jld2" coefs
    
    cs = ChebyshevSeries(coefs,  SA[0.5, -0.2, 1.0], SA[0.7, 0.0, 1.2])
    x̄ = [0.57, -0.02, 1.13]
    x, y, z = x̄

    exy = exp(x*y)
    sxz = sin(x + 0.5*z)
    cxz = cos(x + 0.5*z)
    
    f = exy * cxz
    df_dx = exy * (y*cxz - sxz)
    df_dy = x * exy * cxz
    df_dz = -0.5 * exy * sxz
    d²f_dx² = exy * (y^2*cxz - 2*y*sxz - cxz)
    d²f_dxy = exy * (x*y*cxz - x*sxz + cxz)
    d²f_dxz = -0.5 * exy * (y*sxz + cxz)
    d²f_dy² = x^2 * exy * cxz
    d²f_dyz = -0.5 * x * exy * sxz
    d²f_dz² = -0.25 * exy * cxz
    
    @test cs(x̄) ≈ f

    w, ∇w = gradient(cs, x̄)
    @test w ≈ f
    @test ∇w[1] ≈ df_dx
    @test ∇w[2] ≈ df_dy
    @test ∇w[3] ≈ df_dz

    w, ∇w, Hw = hessian(cs, x̄)
    @test w ≈ f
    @test ∇w[1] ≈ df_dx
    @test ∇w[2] ≈ df_dy
    @test ∇w[3] ≈ df_dz
    @test Hw[1, 1] ≈ d²f_dx²
    @test Hw[1, 2] ≈ d²f_dxy
    @test Hw[1, 3] ≈ d²f_dxz
    @test Hw[2, 1] ≈ d²f_dxy
    @test Hw[2, 2] ≈ d²f_dy²
    @test Hw[2, 3] ≈ d²f_dyz
    @test Hw[3, 1] ≈ d²f_dxz
    @test Hw[3, 2] ≈ d²f_dyz
    @test Hw[3, 3] ≈ d²f_dz²
end

Test Summary:              | Pass  Total  Time
3-D transcedental function |   18     18  1.4s


Test.DefaultTestSet("3-D transcedental function", Any[], 18, false, false, true, 1.778961069782429e9, 1.77896107117094e9, false, "In[10]", Random.Xoshiro(0x7b921d91f7679ffc, 0xe7bb99c6fc402fbf, 0xdebf85c9e7fe77cc, 0xcc805a0c2d446619, 0x2404fe2db0ff50e4))

In [11]:
zeros(2)

2-element Vector{Float64}:
 0.0
 0.0

In [13]:
cs = ChebyshevSeries(zeros(2), SA[-2.0], SA[4.0])
@test contains(cs, 2.0)
@test !contains(cs, 4.1)

Test Passed